# IOBP2 User Data Generation

This notebook generates user data expansion for the IOBP2 dataset following the same pattern as DCLP3, DCLP5, and FLAIR datasets.

## Dataset Overview
IOBP2 (Insulin-Only Bionic Pancreas 2) RCT Public Dataset contains data from a randomized controlled trial comparing:
- BP: Bionic Pancreas (automated insulin delivery)
- BPFiasp: Bionic Pancreas with Fiasp insulin
- Control: Standard of care

The study includes various insulin pumps and uses Dexcom CGM systems.

In [ ]:
import pandas as pd
import numpy as np
import os

## Load IOBP2 Data Files

In [ ]:
# Define data paths
base_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/raw/IOBP2 RCT Public Dataset/Data Tables'

# Load roster data
roster = pd.read_csv(os.path.join(base_path, 'IOBP2PtRoster.txt'), delimiter='|')
print(f"Roster shape: {roster.shape}")
roster.head()

In [ ]:
# Load screening data for device information
screening = pd.read_csv(os.path.join(base_path, 'IOBP2DiabScreening.txt'), delimiter='|')
print(f"Screening shape: {screening.shape}")
print("\nScreening columns:")
print(screening.columns.tolist())

In [ ]:
# Check treatment groups
print("Treatment groups:")
print(roster['TrtGroup'].value_counts())
print("\nUnique pump types in screening:")
print(screening['PumpType'].value_counts())

## Create User Data Expansion DataFrame

Following the same 10-column structure as other datasets:
1. user_id
2. age
3. sex
4. ethnicity
5. bmi
6. cgm_device
7. insulin_delivery_device
8. insulin_delivery_algorithm
9. insulin_delivery_modality
10. tdd

In [ ]:
# Filter completed participants only
completed_roster = roster[roster['RCTPtStatus'] == 'Completed'].copy()
print(f"Completed participants: {len(completed_roster)}")

# Merge roster with screening data
merged_data = completed_roster.merge(screening, on='PtID', how='inner')
print(f"Merged data shape: {merged_data.shape}")

In [ ]:
# Check available demographic and device columns
print("Available columns for mapping:")
relevant_cols = ['PtID', 'AgeAsofEnrollDt', 'TrtGroup', 'Sex', 'Ethnicity', 'Race', 'PumpType', 'CGMUseDevice', 'UnitsInsTotal']
for col in relevant_cols:
    if col in merged_data.columns:
        print(f"{col}: {merged_data[col].dtype}")
        if merged_data[col].dtype == 'object':
            print(f"  Unique values: {merged_data[col].unique()[:10]}")
    else:
        print(f"{col}: NOT FOUND")

In [ ]:
# Create the user data expansion dataframe
user_data_expansion = pd.DataFrame()

# 1. user_id
user_data_expansion['user_id'] = merged_data['PtID']

# 2. age
user_data_expansion['age'] = merged_data['AgeAsofEnrollDt']

# 3. sex
user_data_expansion['sex'] = merged_data['Sex']

# 4. ethnicity - combine ethnicity and race
def combine_ethnicity_race(row):
    ethnicity = str(row['Ethnicity']) if pd.notna(row['Ethnicity']) else ''
    race = str(row['Race']) if pd.notna(row['Race']) else ''
    
    if ethnicity == 'Hispanic or Latino':
        return 'Hispanic/Latino'
    elif race == 'White':
        return 'White'
    elif race == 'Black/African American':
        return 'Black/African American'
    elif race == 'Asian':
        return 'Asian'
    elif race == 'More than one race':
        return 'More than one race'
    else:
        return race if race else 'Unknown'

user_data_expansion['ethnicity'] = merged_data.apply(combine_ethnicity_race, axis=1)

# 5. bmi - Not available in IOBP2, set to NaN
user_data_expansion['bmi'] = np.nan

print("User data expansion created with", len(user_data_expansion), "participants")
print("\nEthnicity distribution:")
print(user_data_expansion['ethnicity'].value_counts())

In [ ]:
# 6. cgm_device - Map CGM devices
def map_cgm_device(cgm_device):
    if pd.isna(cgm_device):
        return 'Dexcom G6'  # Default for IOBP2 timeframe (2019-2021)
    elif 'Dexcom' in str(cgm_device):
        return 'Dexcom G6'  # IOBP2 used Dexcom G6 during study period
    else:
        return str(cgm_device)

user_data_expansion['cgm_device'] = merged_data['CGMUseDevice'].apply(map_cgm_device)

print("CGM device distribution:")
print(user_data_expansion['cgm_device'].value_counts())

In [ ]:
# 7. insulin_delivery_device - Map pump types
def map_insulin_delivery_device(pump_type):
    if pd.isna(pump_type):
        return 'Unknown'
    
    pump_str = str(pump_type).strip()
    
    if 'OmniPod' in pump_str:
        return 'Omnipod'
    elif 'Tandem' in pump_str:
        if 'Control:IQ' in pump_str or 'Control-IQ' in pump_str:
            return 't:slim X2 + Control-IQ'
        elif 'Basal:IQ' in pump_str or 'Basal-IQ' in pump_str:
            return 't:slim X2 + Basal-IQ'
        elif 'X2' in pump_str:
            return 't:slim X2'
        else:
            return 't:slim'
    elif 'Medtronic' in pump_str:
        if '630G' in pump_str:
            return 'MiniMed 630G'
        elif '530G' in pump_str or '551' in pump_str:
            return 'MiniMed 530G'
        else:
            return 'MiniMed'
    elif 'Animas' in pump_str:
        return 'Animas One Touch Ping'
    else:
        return pump_str

user_data_expansion['insulin_delivery_device'] = merged_data['PumpType'].apply(map_insulin_delivery_device)

print("Insulin delivery device distribution:")
print(user_data_expansion['insulin_delivery_device'].value_counts())

In [ ]:
# 8. insulin_delivery_algorithm - Map based on treatment group and device
def map_insulin_delivery_algorithm(trt_group, device):
    if trt_group == 'Control':
        return 'basal-bolus'
    elif trt_group in ['BP', 'BPFiasp']:
        # Bionic Pancreas used automated insulin delivery
        return 'Bionic Pancreas'
    else:
        # Default based on device
        if 'Control-IQ' in str(device):
            return 'Control-IQ'
        elif 'Basal-IQ' in str(device):
            return 'Basal-IQ'
        else:
            return 'basal-bolus'

user_data_expansion['insulin_delivery_algorithm'] = merged_data.apply(
    lambda x: map_insulin_delivery_algorithm(x['TrtGroup'], x['PumpType']), axis=1
)

print("Insulin delivery algorithm distribution:")
print(user_data_expansion['insulin_delivery_algorithm'].value_counts())

In [ ]:
# 9. insulin_delivery_modality - All participants use insulin pumps
user_data_expansion['insulin_delivery_modality'] = 'CSII'

print("Insulin delivery modality distribution:")
print(user_data_expansion['insulin_delivery_modality'].value_counts())

In [ ]:
# 10. tdd - Total daily insulin dose
user_data_expansion['tdd'] = merged_data['UnitsInsTotal']

print("TDD statistics:")
print(user_data_expansion['tdd'].describe())
print(f"\nMissing TDD values: {user_data_expansion['tdd'].isna().sum()}")

In [ ]:
# Final dataframe summary
print("Final IOBP2 user data expansion:")
print(f"Shape: {user_data_expansion.shape}")
print("\nColumns:")
for col in user_data_expansion.columns:
    missing = user_data_expansion[col].isna().sum()
    print(f"{col}: {missing} missing values")

user_data_expansion.head(10)

In [ ]:
# Check data types and basic statistics
print("Data types:")
print(user_data_expansion.dtypes)

print("\nBasic statistics:")
print(user_data_expansion.describe(include='all'))

## Save the Dataset

In [ ]:
# Save to the same folder as other datasets
output_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/user_data_expansion/IOBP2.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

user_data_expansion.to_csv(output_path, index=False)
print(f"IOBP2 user data expansion saved to: {output_path}")
print(f"Final dataset contains {len(user_data_expansion)} participants")